# EduAssist AI: LLMOps Assignment 2
## BITS WILP MLOps: API Cost, Latency, and Prompt Version Management

**Student name:** enter your name  
**BITS ID:** enter your BITS ID

This notebook covers latency measurement, provider-reported token usage, request/monthly cost, three prompt versions, MLflow evaluation, and reviewed production promotion. It reuses the five v1 requests for Tasks 1 and 2 and compares all 15 requests for Task 3.

**Integrity:** No measured results are fabricated. Run the actual API experiment and review the responses before submission. Offline test fixtures are not live evidence. Never put an API key into this notebook or its output.

## 1. MLOps versus LLMOps

MLOps manages datasets, features, training, deployment, and monitoring. LLMOps additionally controls prompts, retrieval context, API/model versions, output contracts, and guardrails. These artifacts can change independently, affecting accuracy, token consumption, cost, and latency. Generated answers are not fully deterministic; a well-formed response can still be incorrect. This solution combines immutable prompt versions, automated JSON/schema checks, and human reference-based review.

Install the pinned requirements in a dedicated Python 3.13 environment and select the EduAssist Assignment 2 kernel. See README.md for commands.

In [1]:
from pathlib import Path
import importlib.metadata
import json
import platform
import sys

candidates = [Path.cwd(), Path.cwd() / 'assignment2']
root = next((candidate.resolve() for candidate in candidates if (candidate / 'config.json').is_file() and (candidate / 'eduassist').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook with assignment2 or the repository root as the working directory.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
from IPython.display import Markdown, display
from eduassist.pipeline import run_experiment
from eduassist.evaluation import latency_summary
from eduassist.prompts import PROMPTS, QUESTIONS
from eduassist.report import finalize

config = json.loads((root / 'config.json').read_text(encoding='utf-8'))
print('Python:', platform.python_version())
print('Libraries:', {name: importlib.metadata.version(name) for name in ['mlflow', 'openai', 'pandas', 'nbconvert', 'reportlab']})
print('Model:', config['model'])
print('API version:', config['api_version'])
print('Pricing:', config['pricing'])
# No environment contents or secrets are displayed.

Python: 3.13.16
Libraries: {'mlflow': '3.11.1', 'openai': '2.24.0', 'pandas': '2.3.1', 'nbconvert': '7.16.6', 'reportlab': '4.5.0'}
Model: gpt-5.4-mini-2026-03-17
API version: OpenAI REST v1 /chat/completions
Pricing: {'currency': 'USD', 'date': '2026-10-31', 'input_per_million': 0.75, 'output_per_million': 4.5, 'cached_input_per_million': 0.075, 'source': 'https://developers.openai.com/api/docs/pricing', 'model_source': 'https://developers.openai.com/api/docs/models/gpt-5.4-mini', 'basis': 'Standard text-token pricing, no tools, no regional uplift; recheck before execution'}


## 2. Experimental Method and Pricing

The fixed evaluation set covers five undergraduate ML/MLOps concepts. All versions use the same question text, model snapshot, completion budget, and reasoning setting. A seed-42 shuffled schedule reduces systematic ordering bias. Each request is non-streaming; UTC start/end timestamps are recorded and `time.perf_counter()` measures end-to-end duration. SDK retries are disabled, and failures are retained. There are no warm-up calls, output repairs, or API-enforced JSON modes.

Defaults: `gpt-5.4-mini-2026-03-17`, OpenAI REST v1 Chat Completions, reasoning effort `none`, 800 maximum completion tokens, default service tier. Published standard USD prices checked on **2026-10-31**: input **0.75**, output **4.50**, cached input **0.075** per million tokens. Recheck prices and model access before running; config.json is the authoritative configuration for your run.

Sources: [OpenAI pricing](https://developers.openai.com/api/docs/pricing), [model documentation](https://developers.openai.com/api/docs/models/gpt-5.4-mini), [MLflow Prompt Registry](https://mlflow.org/docs/latest/genai/prompt-registry/).

In [2]:
display(pd.DataFrame(QUESTIONS))
for label, definition in PROMPTS.items():
    print()
    print(label + ': ' + definition['commit_message'])
    print(definition['template'])

,question_id,question,reference
0,Q1,"What is overfitting in machine learning, and h...",Overfitting fits training noise and generalize...
1,Q2,Explain the difference between supervised and ...,"Supervised learning uses labeled examples, suc..."
2,Q3,"What is gradient descent, and what role does t...",Gradient descent updates parameters opposite t...
3,Q4,Why are precision and recall important for eva...,Precision is TP/(TP+FP); recall is TP/(TP+FN)....
4,Q5,How does LLMOps differ from traditional MLOps?,"MLOps manages data, training, model deployment..."



v1: Basic baseline: clear student question answering without output constraints.
Answer the following student question clearly.
Question: {{question}}

v2: Structured guidance: prioritize concise, accurate undergraduate explanations.
You are an academic assistant.
Answer the following student question in a clear, concise, and accurate manner.
Use simple language suitable for undergraduate students.
Question: {{question}}

v3: JSON contract: answer, confidence enum, and string key points without markdown.
You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: {{question}}


## 3. Execute the API Evaluation

Set `OPENAI_API_KEY` privately in the environment of the terminal/kernel. Do not assign a literal key in code. Either run `python -m eduassist.pipeline` from assignment2 **or** enable the flag below. This makes **15 paid requests**. Keep the flag false when viewing/exporting existing results. Existing outputs are protected unless overwrite is deliberately enabled; archive them first. If API access is unavailable, this notebook does not invent responses. The handout permits a real local model with documented tokenizer/hosting costs, which requires a separate adaptation.

In [3]:
RUN_LIVE = False
OVERWRITE_RESULTS = False

if RUN_LIVE:
    manifest = run_experiment(root=root, overwrite=OVERWRITE_RESULTS)
else:
    print('No API call made. Use python -m eduassist.pipeline or deliberately enable RUN_LIVE after private key setup.')

No API call made. Use python -m eduassist.pipeline or deliberately enable RUN_LIVE after private key setup.


In [4]:
request_path = root / 'latency_tokens_cost.csv'
comparison_path = root / 'prompt_evaluation_results.csv'
requests = pd.read_csv(request_path) if request_path.is_file() else pd.DataFrame()
comparison = pd.read_csv(comparison_path) if comparison_path.is_file() else pd.DataFrame()
manifest_path = root / 'outputs' / 'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.is_file() else None
if manifest is not None:
    print('Evaluation batch ID:', manifest['batch_id'])
    print('Actual registry versions:', manifest['registry_versions'])
if requests.empty:
    print('Real results pending. No synthetic measurements are substituted.')
else:
    print('Recorded requests:', len(requests))
    display(requests[['question_id', 'prompt_version', 'status', 'model_name', 'run_id']])

Real results pending. No synthetic measurements are substituted.


## Task 1: API Latency (Five Requests)

The five basic v1 requests provide the Task 1 table. Summary statistics include all attempts, so failures do not disappear from latency evidence. Standard deviation uses the sample definition (`ddof=1`). End-to-end latency is not time-to-first-token or a pure model-compute measurement.

In [5]:
if not requests.empty:
    baseline = requests[requests['prompt_version'] == 'v1'].sort_values('question_id')
    display(baseline[['question_id', 'prompt_version', 'model_name', 'start_time_utc', 'end_time_utc', 'latency_seconds', 'status']])
    display(pd.DataFrame([latency_summary(baseline.to_dict('records'))]))
    slowest = baseline.loc[baseline['latency_seconds'].idxmax()]
    display(Markdown(f"The slowest v1 request was **{slowest['question_id']}** at **{slowest['latency_seconds']:.3f} seconds**, status **{slowest['status']}**, with **{slowest['output_tokens']}** reported output tokens. Longer generation, provider queuing, connection setup, or network variability may contribute. These five observations do not isolate the cause; token length alone cannot prove it."))
else:
    print('Run the real experiment to populate the five-request latency table and interpretation.')

Run the real experiment to populate the five-request latency table and interpretation.


## Task 2: Token Consumption and Cost

$$C_{in}=\frac{T_{in}}{1{,}000{,}000}P_{in},\qquad C_{out}=\frac{T_{out}}{1{,}000{,}000}P_{out}$$
$$C_{request}=C_{in}+C_{out},\qquad C_{month}=\overline{C_{request}}\times10{,}000$$

Tokens come from API usage, not character counts or guesses. Output usage includes billed reasoning tokens if any. `total_cost_usd` uses the assignment's standard-input formula; the separate `cache_adjusted_cost_usd` discounts actual cached input tokens and is closer to a token-only invoice estimate. Unknown costs remain blank, not zero. Token/cost averages use responses with known usage; incomplete versions cannot pass promotion. Monthly projections assume the same workload mix and exclude taxes, infrastructure, tools, and unobserved retries.

In [6]:
if not requests.empty:
    display(requests[['question_id', 'prompt_version', 'input_tokens', 'output_tokens', 'total_tokens', 'cached_input_tokens', 'reasoning_tokens', 'input_cost_usd', 'output_cost_usd', 'total_cost_usd', 'cache_adjusted_cost_usd']])
if not comparison.empty:
    monthly = comparison[['prompt_version', 'avg_cost_per_request']].copy()
    monthly['requests_per_month'] = 10_000
    monthly['projected_monthly_cost_usd'] = monthly['avg_cost_per_request'] * monthly['requests_per_month']
    display(monthly)
display(Markdown('**Cost optimization:** shorten answers and remove redundant context while preserving the required schema and academic quality. Output tokens have a higher unit price, so concise responses can lower cost and generation time. Treat any changed prompt as a new version and evaluate it before adoption; do not assume prompt caching savings for short requests.'))

**Cost optimization:** shorten answers and remove redundant context while preserving the required schema and academic quality. Output tokens have a higher unit price, so concise responses can lower cost and generation time. Treat any changed prompt as a new version and evaluate it before adoption; do not assume prompt caching savings for short requests.

## Task 3: Prompt Versioning, Evaluation, and Promotion

Experiment: **eduassist_llmops_assignment**. Prompt Registry name: **eduassist_academic_qa**. The templates have meaningful commit messages and immutable versions. A new database creates versions 1-3; subsequent evaluations append later versions. Actual URIs, versions, and run IDs are preserved so assignment labels v1-v3 are never confused with registry version numbers.

$$\mathrm{valid\_json}=\frac{\text{strictly valid JSON responses}}{\text{all five evaluated requests}}$$

`valid_json` is the aggregate of per-response boolean checks; failed/refused/truncated/missing-usage requests fail this evaluation. A JSON array or object with incorrect fields can be syntactically valid but unusable. `schema_valid_rate` additionally requires exactly `answer` (string), `confidence` (high/medium/low), and `key_points` (array of strings). No fences are stripped, no JSON is repaired, and NaN/Infinity/duplicate keys are rejected. An empty key_points array is allowed by the supplied schema; usefulness is assessed by the human review.

In [7]:
if not comparison.empty:
    columns = ['prompt_version', 'registry_version', 'avg_latency_seconds', 'avg_input_tokens', 'avg_output_tokens', 'avg_cost_per_request', 'projected_monthly_cost_usd', 'valid_json', 'schema_valid_rate', 'recommended']
    display(comparison[columns])
    if 'avg_quality_score' in comparison.columns:
        display(comparison[['prompt_version', 'avg_quality_score', 'quality_pass_rate']])
else:
    print('Prompt comparison pending real evaluation.')
diff_path = root / 'outputs' / 'prompt_diff.txt'
if diff_path.is_file():
    print(diff_path.read_text(encoding='utf-8'))

Prompt comparison pending real evaluation.


## Human Quality Review and Production Gate

Complete all 15 rows in **human_review.csv** against the reference concepts. Edit only the score, reviewer, and notes fields; do not change recorded answers or identifiers. Accuracy: 0 incorrect, 1 partly correct, 2 conceptually correct. Clarity: 0 confusing, 1 understandable but awkward/verbose, 2 clear and concise. Relevance: 0 off-topic, 1 partial, 2 directly addresses the question. Include reviewer and a concrete explanatory note for every row.

An eligible version must have five successful complete responses, five schema-compliant responses, and accuracy=2, relevance=2, clarity>=1 for every answer. Among eligible versions choose lowest mean cost, then latency. A cheaper prose response is not a production candidate for a strict JSON interface. If none pass, do not promote. Self-reported confidence is not a calibrated quality measure.

After genuine review, run **python -m eduassist.report --promote** or deliberately enable the following flag. The production alias is set and reloaded for verification. The generated 500-700 word REPORT.pdf includes a 150+ word measured promotion justification covering quality, JSON, latency, tokens, and cost.

In [8]:
PROMOTE_AFTER_REVIEW = False

if PROMOTE_AFTER_REVIEW:
    manifest = finalize(root=root, promote=True)
else:
    print('No promotion performed by this cell. Complete human_review.csv, then run python -m eduassist.report --promote.')

if comparison_path.is_file():
    comparison = pd.read_csv(comparison_path)
    display(comparison)
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('Production promoted:', manifest['production_promoted'])
    print('Production registry version:', manifest['production_version'])
    if PROMOTE_AFTER_REVIEW:
        import mlflow.genai
        production = mlflow.genai.load_prompt(f"prompts:/{manifest['config']['prompt_name']}@production", cache_ttl_seconds=0)
        print('Verified production prompt URI:', f'prompts:/{production.name}/{production.version}')

No promotion performed by this cell. Complete human_review.csv, then run python -m eduassist.report --promote.


In [9]:
report_path = root / 'REPORT.md'
if report_path.is_file():
    display(Markdown(report_path.read_text(encoding='utf-8')))
    checks_path = root / 'outputs' / 'report_checks.json'
    if checks_path.is_file():
        print('Report checks:', json.loads(checks_path.read_text(encoding='utf-8')))
else:
    print('Final report pending real measurements and completed human review.')

Final report pending real measurements and completed human review.


## Submission Evidence and Reproduction

Start the local MLflow UI with `python -m eduassist.ui`; open http://127.0.0.1:5000. If occupied, use `--port 5001`. Filter experiment runs to the manifest batch ID. Capture real screenshots in screenshots/ with these exact filenames:

- experiment.png: named experiment and the three current runs.
- prompt_registry.png: the three actual prompt versions and commit messages.
- metrics.png: run comparison showing every required metric.
- prompt_diff.png: Prompt Registry diff between versions.
- production_alias.png: selected version with its production alias.

After the real CLI run and promotion, keep RUN_LIVE and PROMOTE_AFTER_REVIEW false, execute all notebook cells, and save their measured outputs. Export using `python -m nbconvert --to html EduAssist_LLMOps_Assignment2.ipynb`. Build the evidence-gated ZIP with `python -m eduassist.package`. The ZIP includes the executable notebook, HTML export, requirements.txt, latency_tokens_cost.csv, prompt_evaluation_results.csv, genuine review/evidence, five screenshots, final REPORT.pdf, README, config, code, and tests.

Before submitting: replace name/ID placeholders, read and personalize the report, verify the measured comparison and alias, inspect screenshots for secrets, and follow BITS rules on acknowledging AI assistance. Five questions per version are assignment-scale observations, not a production SLA or proof of future correctness.